# Linear Regression

In [ ]:
# Import the required libraries



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Data Preprocessing

### **Exploring the dataset**

Let's start with loading the training data from the csv into a pandas dataframe



Load the datasets from GitHub. Train dataset has already been loaded for you in df below. To get test dataset use the commented code.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/cronan03/DevSoc_AI-ML/main/train_processed_splitted.csv')

Let's see what the first 5 rows of this dataset looks like

In [ ]:
df.head()

,LotArea,TotalBsmtSF,GrLivArea,GarageArea,PoolArea,OverallCond,Utilities,SalePrice
0,11553,1051,1159,336,0,5,AllPub,158000
1,8400,1052,1052,288,0,5,AllPub,138500
2,8960,1008,1028,360,0,6,AllPub,115000
3,11100,0,930,308,0,7,AllPub,84900
4,15593,1304,2287,667,0,4,AllPub,225000


What are all the features present? What is the range for each of the features along with their mean?

In [ ]:

print("Features/Columns present in dataset:")
print(list(df.columns))


stats = df.describe().T[['mean', 'min', 'max']]
stats['range'] = stats['max'] - stats['min']


stats


Features/Columns present in dataset:
['LotArea', 'TotalBsmtSF', 'GrLivArea', 'GarageArea', 'PoolArea', 'OverallCond', 'Utilities', 'SalePrice']


,mean,min,max,range
LotArea,10622.104262,1300.0,215245.0,213945.0
TotalBsmtSF,1058.311263,0.0,6110.0,6110.0
GrLivArea,1512.900304,334.0,5642.0,5308.0
GarageArea,473.480213,0.0,1418.0,1418.0
PoolArea,2.643075,0.0,738.0,738.0
OverallCond,5.582192,1.0,9.0,8.0
SalePrice,180795.504566,34900.0,755000.0,720100.0


### **Feature Scaling and One-Hot Encoding**

You must have noticed that some features `(such as Utilities)`


are not continuous values.
  
These features contain values indicating different categories and must somehow be converted to numbers so that the computer can understand it. `(Computers only understand numbers and not strings)`
  
These features are called categorical features. We can represent these features as a `One-Hot Representation`
  
  
You must have also noticed that all the other features, each are in a different scale. This can be detremental to the performance of our linear regression model and so we normalize them so that all of them are in the range $[0,1]$

> NOTE: When you are doing feature scaling, store the min/max which you will use to normalize somewhere. This is then to be used at testing time. Try to think why are doing this?

In [ ]:
# One-hot encode the categorical feature (Utilities)
df = pd.get_dummies(df, columns=['Utilities'], dtype='int32')
df.head()

In [ ]:
# Min-max scale every continuous column (incl. SalePrice) to [0, 1].
# One-hot columns are already 0/1. Store train min/max for use at test time.
num_cols = ['LotArea', 'TotalBsmtSF', 'GrLivArea', 'GarageArea', 'PoolArea', 'OverallCond', 'SalePrice']
col_min = df[num_cols].min()
col_max = df[num_cols].max()
df[num_cols] = (df[num_cols] - col_min) / (col_max - col_min)
df.describe().T[['mean', 'min', 'max']]

**Why store min/max?** At test time the model expects inputs on exactly the same scale it was trained on. Using the training min/max keeps a given raw value mapped to the same scaled value; recomputing from the test set would shift the mapping and also leak test information into preprocessing. The same min/max (for `SalePrice`) is used to convert predictions back to dollars.

### **Conversion to NumPy**

Ok so now that we have all preprocessed all the data, we need to convert it to numpy for our linear regression model
  
Assume that our dataset has a total of $N$ datapoints. Each datapoint having a total of $D$ features (after one-hot encoding), we want our numpy array to be of shape $(N, D)$

In our task, we have to predict the `SalePrice`. We will need 2 numpy arrays $

*   List item
*   List item

(X, Y)$. These represent the features and targets respectively

In [ ]:
# Convert to numpy array
x = df.drop('SalePrice', axis=1).to_numpy(dtype=float)       # (N, D)
y = df['SalePrice'].to_numpy(dtype=float).reshape(-1, 1)     # (N, 1)
print(x.shape, y.shape)

## Linear Regression formulation
  
We now have our data in the form we need. Let's try to create a linear model to get our initial (Really bad) prediction


Let's say a single datapoint in our dataset consists of 3 features $(x_1, x_2, x_3)$, we can pose it as a linear equation as follows:
$$ y = w_1x_1 + w_2x_2 + w_3x_3 + b $$
Here we have to learn 4 parameters $(w_1, w_2, w_3, b)$
  
  
Now how do we extend this to multiple datapoints?  
  
  
Try to answer the following:
- How many parameters will we have to learn in the cae of our dataset? (Don't forget the bias term)
- Form a linear equation for our dataset. We need just a single matrix equation which correctly represents all the datapoints in our dataset
- Implement the linear equation as an equation using NumPy arrays (Start by randomly initializing the weights from a standard normal distribution)

**Parameters:** D weights + 1 bias = D + 1 (here D = 8: 6 numeric features + 2 Utilities dummies, so 9 parameters).

**Matrix equation:** $\hat{Y} = XW^T + b$, with $X \in \mathbb{R}^{N\times D}$, $W \in \mathbb{R}^{1\times D}$, $b$ broadcast over all $N$ rows.

In [ ]:
N, D = x.shape
# Parameters to learn: D weights + 1 bias = D + 1
# Matrix equation: Y_hat = X @ W^T + b, with X: (N, D), W: (1, D), b: scalar (broadcast over N)
np.random.seed(0)
w = np.random.randn(1, D)
b = np.random.randn(1)
y_pred = x @ w.T + b
print('Number of parameters:', w.size + b.size)

How well does our model perform? Try comparing our predictions with the actual values

In [ ]:
print('Predicted (first 5):', y_pred[:5].squeeze().tolist())
print('Actual    (first 5):', y[:5].squeeze().tolist())
print('Initial MSE:', np.mean((y - y_pred) ** 2))

### **Learning weights using gradient descent**

So these results are really horrible. We need to somehow update our weights so that it correclty represents our data. How do we do that?

We must do the following:
- We need some numerical indication for our performance, for this we define a Loss Function ( $\mathscr{L}$ )
- Find the gradients of the `Loss` with respect to the `Weights`
- Update the weights in accordance to the gradients: $W = W - \alpha\nabla_W \mathscr{L}$

Lets define the loss function:
- We will use the MSE loss since it is a regression task. (Specify the assumptions we make while doing so as taught in the class).
- Implement this loss as a function. (Use numpy as much as possible)

**MSE assumptions:** $\mathscr{L} = \frac{1}{N}\sum_i (y_i-\hat y_i)^2$ assumes the target is a linear function of the features plus Gaussian noise that is zero-mean, with constant variance (homoscedastic) and independent across samples. Under these assumptions minimising MSE is the same as maximum-likelihood estimation. MSE is also sensitive to outliers.

**Gradients:** $\nabla_W\mathscr{L} = \frac{2}{N}(\hat Y - Y)^T X$ and $\frac{\partial\mathscr{L}}{\partial b} = \frac{2}{N}\sum_i(\hat y_i-y_i)$.

In [ ]:
def mse_loss_fn(y_true, y_pred):
    """Mean squared error: (1/N) * sum (y_true - y_pred)^2"""
    return np.mean((y_true - y_pred) ** 2)

Calculate the gradients of the loss with respect to the weights (and biases). First write the equations down on a piece of paper, then proceed to implement it

In [ ]:
def get_gradients(y_true, y_pred, W, b, X):
    """
    Calculates the gradients for the MSE loss function with respect to the weights (and bias)

    Args:
        y_true: The true values of the target variable (SalePrice in our case)
        y_pred: The predicted values of the target variable using our model (W*X + b)

        W: The weights of the model
        b: The bias of the model
        X: The input features

    Returns:
        dW: The gradients of the loss function with respect to the weights
        db: The gradients of the loss function with respect to the bias
    """
    N = X.shape[0]
    err = y_pred - y_true                 # (N, 1)
    dW = (2 / N) * (err.T @ X)            # (1, D)
    db = (2 / N) * np.sum(err)            # scalar
    return dW, db

Update the weights using the gradients

In [ ]:
def update(weights, bias, gradients_weights, gradients_bias, lr):
    """
    Updates the weights (and bias) using the gradients and the learning rate

    Args:
        weights: The current weights of the model
        bias: The current bias of the model

        gradients_weights: The gradients of the loss function with respect to the weights
        gradients_bias: The gradients of the loss function with respect to the bias

        lr: The learning rate

    Returns:
        weights_new, bias_new: The updated weights and bias of the model

    """
    weights_new = weights - lr * gradients_weights
    bias_new = bias - lr * gradients_bias
    return weights_new, bias_new

Put all these together to find the loss value, its gradient and finally updating the weights in a loop. Feel free to play around with different learning rates and epochs
  
> NOTE: The code in comments are just meant to be used as a guide. You will have to do changes based on your code

In [ ]:
NUM_EPOCHS = 1000
LEARNING_RATE = 2e-2

losses = []

for epoch in range(NUM_EPOCHS):
    y_pred = x @ w.T + b
    loss = mse_loss_fn(y, y_pred)
    losses.append(loss)
    dw, db = get_gradients(y, y_pred, w, b, x)
    w, b = update(w, b, dw, db, LEARNING_RATE)


Now use matplotlib to plot the loss graph

In [ ]:
plt.plot(losses)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

### **Testing with test data**

Load and apply all the preprocessing steps used in the training data for the testing data as well. Remember to use the **SAME** min/max values which you used for the training set and not recalculate them from the test set. Also mention why we are doing this.

To load test data from GitHub, use the code below.


In [ ]:
df_test = pd.read_csv('https://raw.githubusercontent.com/cronan03/DevSoc_AI-ML/main/test_processed_splitted.csv')
print(df_test)

# One-hot encode Utilities in the test set the same way as in training
df_test = pd.get_dummies(df_test, columns=['Utilities'], dtype='int32')

# Any dummy column present in train but missing in test (e.g. Utilities_NoSeWa) gets all zeros,
# and the column order is forced to match the training dataframe.
missing_cols = set(df.columns) - set(df_test.columns)
print('Columns added with zeros:', missing_cols)
df_test = df_test.reindex(columns=df.columns, fill_value=0)

Using the weights learnt above, predict the values in the test dataset. Also answer the following questions:
- Are the predictions good?
- What is the MSE loss for the testset
- Is the MSE loss for testing greater or lower than training
- Why is this the case

**Answers (check against your own run's numbers):**
- **Are the predictions good?** Moderately: a linear model on 6 raw features captures the main trend (e.g. living area), but it will miss non-linear effects and outliers, so individual predictions can be off by tens of thousands of dollars.
- **Test MSE:** printed above as `Test Loss` (in scaled units; the dollar RMSE is printed in the prediction cell).
- **Test vs. train:** usually slightly higher, since the weights were fit to the training data. It can occasionally come out lower if the test split happens to contain easier/fewer outlier houses.
- **Why:** the model only optimised the training loss, so it has seen the training points but not the test ones (generalisation gap).

In [ ]:
# Fill NaN values
df_test.fillna(0, inplace=True)

# Scale features using the TRAIN min/max (not recomputed from the test set)
df_test[num_cols] = (df_test[num_cols] - col_min) / (col_max - col_min)

# Check for unexpected NaNs
print('NaNs in test set:', df_test.isna().sum().sum())

# Convert to numpy array
x_test = df_test.copy().drop('SalePrice', axis=1).to_numpy(dtype=float) # (N, D)
y_test = df_test.copy()['SalePrice'].to_numpy(dtype=float).reshape(-1, 1) # (N, 1)
print(x_test.shape)

In [ ]:
extra_cols = list(set(df_test.columns) - set(df.columns))
print("Extra columns in df_test:", extra_cols)

missing_cols = list(set(df.columns) - set(df_test.columns))
print("Missing columns in df_test:", missing_cols)

In [ ]:
# Make predictions
y_pred_test = x_test @ w.T + b # (N, 1)
loss_test = mse_loss_fn(y_test, y_pred_test)

# Scale the predictions (and targets) back to the original scale
smin, smax = col_min['SalePrice'], col_max['SalePrice']
y_pred_test_scaled = y_pred_test * (smax - smin) + smin
y_test_scaled = y_test * (smax - smin) + smin

print('Train loss (scaled):', losses[-1])
print('Test loss (scaled): ', loss_test)
print('Test RMSE in dollars:', np.sqrt(np.mean((y_test_scaled - y_pred_test_scaled) ** 2)))

In [ ]:
idx = np.random.randint(0, x_test.shape[0], 5)
y_pred_test_sample = y_pred_test_scaled[idx].round().astype(int)
y_true_test_sample = y_test_scaled[idx].round().astype(int)

print('Predicted SalePrice: \t', y_pred_test_sample.squeeze().tolist())
print('Actual SalePrice: \t', y_true_test_sample.squeeze().tolist())
print('\nTest Loss: \t\t', loss_test)